In [1]:
import jax_sn as jsn
%load_ext autoreload
%autoreload 2
import jax_sbgeom
from jax_sbgeom.flux_surfaces import ToroidalExtent
from jax_sbgeom.jax_utils import mesh_to_pyvista_mesh

import jax.numpy as jnp
import numpy as onp
from typing import Tuple
import jax
import numpy as onp
from functools import partial
jax.config.update("jax_enable_x64", True)

import pyvista as pv

def remove_zeros(array):
    a = onp.array(array)
    a[onp.abs(a) < 1e-14] = 0.0
    return a

from jax_sn.domain.geometric_domain import GeometricDomain, TopologicalSortedAngle
from jax_sn.solution_domain.solution_domain import FEMMatrices, PrecomputedGeometricData
from jax_sn.domain.geometric_domain import CrossSectionData, CrossSectionDataSingleGroup
from jax_sn.source import Source
from typing import List
from jax_sn.tools.mesh_tools import _pad_last_dimension
onp.set_printoptions(linewidth = 200)

In [2]:
from jax_sn.domain.domain_generation import create_cartesian_setup
from jax_sn.solution_domain import BasixLagrangianSimplex
from jax_sn.solution_domain import SolutionDomain
from jax_sn.domain import CrossSectionData, Domain

In [28]:
from jax_sn.solution_domain import compute_setup_for_solution_domain, transport_inverse_sweep_complete_vmap_fori
from jax_sn.solution_domain.transport_operator import transport_inverse_sweep_multiple_angles_batched, transport_inverse_sweep_multiple_angles_batched_single_f


sigma  = 5
xs_0   = CrossSectionData.constant(sigma, 1, 1)
q      = 5
dim    = 3
degree = 2
n_elem = 10
verts, conn, cross_sections, sources = create_cartesian_setup(dim, [[(1.0, n_elem)], [(0.2, 10)], [(0.2, 10)]], cross_sections_regions=[xs_0], source_regions=[q], inhomogeneity_direction=0)
element_type                         = BasixLagrangianSimplex(degree, dim)
domain                               = Domain.from_mesh_and_cross_sections((verts, conn), element_type.face_template, cross_sections_indexed=cross_sections)
solution_domain                      = SolutionDomain.from_element_and_domain(element_type, domain)
source_mapped                        = Source(jnp.repeat(sources, element_type.n_basis, axis=-1).reshape(-1, element_type.n_basis))
buffer_sweep                         = jnp.zeros((domain.n_elements, element_type.n_basis))

quad_set = jsn.quadrature_set.create_tn_quadrature_set(4)
toposorts = [TopologicalSortedAngle.from_domain_angle(domain, angle, 100) for angle in quad_set.angles]
total_toposorts = jax.tree.map(lambda * xs : jnp.stack(xs), *toposorts)

In [29]:
source_angle = jax.tree.map(lambda ts : jnp.repeat(ts, total_toposorts.angle.shape[0], axis=0).reshape((total_toposorts.angle.shape[0],)+ts.shape), source_mapped)

In [30]:
total_buffer_sweep = jnp.zeros((total_toposorts.angle.shape[0], domain.n_elements, element_type.n_basis))

In [31]:
%timeit transport_inverse_sweep_complete_vmap_fori(*solution_domain.solution_domain_data, solution_domain.map_cross_sections_to_full_array(0), toposorts[0], source_mapped, buffer_sweep).block_until_ready()

61.7 ms ± 14 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [35]:
transport_inverse_sweep_multiple_angles_batched_single_f(*solution_domain.solution_domain_data, solution_domain.map_cross_sections_to_full_array(0), total_toposorts, source_angle, total_buffer_sweep, 500).block_until_ready()

Array([[[ 6.35762658e-01,  6.35762054e-01,  5.97072186e-01, ...,  6.35762270e-01,  6.16896581e-01,  6.35763148e-01],
        [ 6.35760945e-01,  5.97062140e-01,  6.35760527e-01, ...,  6.16896407e-01,  6.35760569e-01,  6.16897016e-01],
        [ 6.35762763e-01,  6.35761469e-01,  5.97072211e-01, ...,  6.35762836e-01,  6.16896146e-01,  6.35763214e-01],
        ...,
        [ 9.16825046e-02, -1.00080787e-03, -1.08608069e-03, ...,  4.86224247e-02,  4.97320608e-02,  4.60668122e-02],
        [ 8.99216325e-02, -1.35980064e-02,  5.83956270e-02, ...,  4.92601810e-02,  1.00986084e-01,  3.96283056e-02],
        [ 3.54395791e-06, -1.38177590e-02,  5.79427214e-02, ...,  3.54395791e-06,  3.53429284e-02, -1.36632277e-03]],

       [[ 4.90759935e-06,  4.96471884e-06,  9.60439882e-02, ...,  4.63346343e-06,  4.92454645e-02,  4.63356351e-06],
        [ 1.88393454e-05,  9.60579119e-02,  1.94001459e-05, ...,  4.92418873e-02,  1.85054300e-05,  4.92420369e-02],
        [ 4.71121032e-06,  4.71715881e-06,  9.604

In [36]:
from jax_sn.solution_domain.transport_operator import transport_inverse_sweep_multiple_angles_batched, transport_inverse_sweep_multiple_angles_batched_single_f

In [37]:
from finite_elements.test_transport_operator import _expected_values_from_sb, _matfree_nte_from_sb, _test_domain_from_sb

In [39]:
%timeit simulation.simulation.Total_Sweep(simulation.energy_set[0], source)

154 ms ± 1.62 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
